# 공통 코드

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import MinMaxScaler

pd.set_option('display.max_rows', None)      # 모든 행 출력
pd.set_option('display.max_columns', None)   # 모든 열 출력
pd.set_option('display.width', None)         # 출력 너비 제한 해제
pd.set_option('display.max_colwidth', None)  # 컬럼 내용 길이 제한 해제

In [8]:
# 입력 데이터
DATA_DIR = Path("processed_data/psf_tables/number")

# GMM 저장 위치
GMM_DIR = Path("models/gmm")
GMM_DIR.mkdir(parents=True, exist_ok=True)

In [9]:
# 공통 컬럼 정리 함수
def clean_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace("\u00a0", " ", regex=False)
        .str.strip()
    )
    return df

In [10]:
def make_composite_state(
    df,
    group_cols,
    state_name,
    n_components=3,
    random_state=42,
):
    result = df.copy()

    X = result[group_cols].copy()

    X = X.apply(
        pd.to_numeric,
        errors="coerce",
    )

    # API에서도 동일하게 사용할 결측치 대체값
    fill_values = {}

    for col in group_cols:
        if X[col].isna().any():

            mode = X[col].mode(
                dropna=True
            )

            if mode.empty:
                raise ValueError(
                    f"{col}의 유효값이 없습니다."
                )

            fill_value = mode.iloc[0]

            X[col] = X[col].fillna(
                fill_value
            )

            fill_values[col] = float(
                fill_value
            )

        else:
            # 결측이 없더라도 API용으로 최빈값 저장
            mode = X[col].mode(
                dropna=True
            )

            if not mode.empty:
                fill_values[col] = float(
                    mode.iloc[0]
                )

    # ----------------------------------------
    # 단일 변수
    # ----------------------------------------

    if len(group_cols) == 1:

        col = group_cols[0]

        result[state_name] = X[col]

        return {
            "result": result,
            "gmm": None,
            "label_map": None,
            "group_cols": group_cols,
            "fill_values": fill_values,
        }

    # ----------------------------------------
    # GMM
    # ----------------------------------------

    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type="full",
        random_state=random_state,
        n_init=20,
    )

    cluster = gmm.fit_predict(X)

    cluster_mean = pd.DataFrame(
        gmm.means_,
        columns=group_cols,
    )

    cluster_mean["mean_score"] = (
        cluster_mean[group_cols]
        .mean(axis=1)
    )

    ordered_clusters = (
        cluster_mean
        .sort_values("mean_score")
        .index
        .tolist()
    )

    label_map = {
        int(cluster_id): int(label)
        for label, cluster_id
        in enumerate(ordered_clusters)
    }

    result[state_name] = [
        label_map[int(c)]
        for c in cluster
    ]

    result[
        state_name + "_cluster"
    ] = cluster

    return {
        "result": result,
        "gmm": gmm,
        "label_map": label_map,
        "group_cols": list(group_cols),
        "fill_values": fill_values,
        "cluster_mean": cluster_mean,
    }

In [20]:
GMM_GROUPS = {

    "PSF2": {
        "Equipment_Quality": [
            "작업 도구의 적절성",
            "장비의 사용성",
        ],

        "System_Support": [
            "컴퓨터 지원 시스템의 적절성",
            "타 기기와의 연계성",
        ],
    },

    "PSF4": {
        "Training_Frequency": [
            "교육 주기(빈도)",
            "직무능력 평가 빈도",
        ],

        "Technical_Knowledge": [
            "시스템 지식(원리적 지식)의 숙련도",
            "장비 사용 지식의 숙련도",
        ],
    },

    "PSF6": {
        "Thermal_Comfort": [
            "실내 온도",
            "실내 습도",
            "통풍",
        ],

        "Visual_Environment": [
            "업무공간 가시성",
            "안개",
            "황사",
        ],

        "Surface_Safety": [
            "마찰계수",
            "비",
        ],

        "Outdoor_Climate": [
            "실외 온도",
            "실외 습도",
            "풍속",
            "낙뢰",
            "비",
        ],
    },

    "PSF8": {
        "Decision_Complexity": [
            "의사결정 대안의 수",
            "의사결정 판단 기준의 수",
        ],

        "Information_Complexity": [
            "관측되는 정보의 양",
            "(관측이 불가능하여)추론해야하는 정보의 양",
        ],
    },
}

In [21]:
def build_and_save_psf_gmm(
    df,
    psf_name,
    group_config,
):

    bundle = {}

    working_df = df.copy()

    for state_name, group_cols in group_config.items():

        print(
            f"[{psf_name}] {state_name}"
        )

        output = make_composite_state(
            df=working_df,
            group_cols=group_cols,
            state_name=state_name,
        )

        working_df = output[
            "result"
        ]

        bundle[state_name] = {
            "gmm": output["gmm"],
            "label_map": output["label_map"],
            "group_cols": output["group_cols"],
            "fill_values": output["fill_values"],
        }

        if output["gmm"] is not None:

            print(
                " label_map:",
                output["label_map"],
            )

            display(
                output["cluster_mean"]
            )

    save_path = (
        GMM_DIR
        / f"{psf_name.lower()}_gmm.joblib"
    )

    joblib.dump(
        bundle,
        save_path,
    )

    print(
        f"\n저장 완료: {save_path}"
    )

    return working_df, bundle

In [22]:
psf2_df = pd.read_csv(
    DATA_DIR / "장비_및_기기.csv"
)

psf2_result, psf2_bundle = (
    build_and_save_psf_gmm(
        psf2_df,
        "PSF2",
        GMM_GROUPS["PSF2"],
    )
)

[PSF2] Equipment_Quality
 label_map: {0: 0, 1: 1, 2: 2}


,작업 도구의 적절성,장비의 사용성,mean_score
0,0.000000,0.543849,0.271925
1,1.492205,0.517118,1.004662
2,1.038121,2.000000,1.519060


[PSF2] System_Support
 label_map: {1: 0, 2: 1, 0: 2}


,컴퓨터 지원 시스템의 적절성,타 기기와의 연계성,mean_score
0,0.978368,2.0,1.489184
1,1.012808,0.0,0.506404
2,0.999047,1.0,0.999523



저장 완료: models\gmm\psf2_gmm.joblib


In [23]:
psf4_df = pd.read_csv(
    DATA_DIR / "교육_및_훈련.csv"
)

psf4_result, psf4_bundle = (
    build_and_save_psf_gmm(
        psf4_df,
        "PSF4",
        GMM_GROUPS["PSF4"],
    )
)

[PSF4] Training_Frequency
 label_map: {1: 0, 2: 1, 0: 2}


,교육 주기(빈도),직무능력 평가 빈도,mean_score
0,2.0,0.93627,1.468135
1,0.0,1.01030,0.505150
2,1.0,1.02495,1.012475


[PSF4] Technical_Knowledge
 label_map: {0: 0, 2: 1, 1: 2}


,시스템 지식(원리적 지식)의 숙련도,장비 사용 지식의 숙련도,mean_score
0,0.0,0.967864,0.483932
1,2.0,1.000000,1.500000
2,1.0,0.979962,0.989981



저장 완료: models\gmm\psf4_gmm.joblib


In [24]:
psf6_df = pd.read_csv(
    DATA_DIR / "환경.csv"
)

psf6_result, psf6_bundle = (
    build_and_save_psf_gmm(
        psf6_df,
        "PSF6",
        GMM_GROUPS["PSF6"],
    )
)

[PSF6] Thermal_Comfort
 label_map: {1: 0, 2: 1, 0: 2}


,실내 온도,실내 습도,통풍,mean_score
0,0.656754,1.0,1.000000,0.885585
1,0.659785,0.0,0.503591,0.387792
2,0.689142,1.0,0.000000,0.563047


[PSF6] Visual_Environment
 label_map: {1: 0, 0: 1, 2: 2}


,업무공간 가시성,안개,황사,mean_score
0,0.507861,1.0,1.0,0.835954
1,1.004132,1.0,0.0,0.668044
2,2.000000,1.0,1.0,1.333333


[PSF6] Surface_Safety
 label_map: {1: 0, 0: 1, 2: 2}


,마찰계수,비,mean_score
0,0.000000,1.0,0.500000
1,0.491176,0.0,0.245588
2,1.000000,1.0,1.000000


[PSF6] Outdoor_Climate
 label_map: {1: 0, 2: 1, 0: 2}


,실외 온도,실외 습도,풍속,낙뢰,비,mean_score
0,0.515954,1.0,0.473811,1.0,0.500903,0.698134
1,0.000000,0.0,0.509050,1.0,0.513575,0.404525
2,1.000000,0.0,0.497748,1.0,0.503378,0.600225



저장 완료: models\gmm\psf6_gmm.joblib


In [25]:
psf8_df = pd.read_csv(
    DATA_DIR / "복잡성.csv"
)

psf8_result, psf8_bundle = (
    build_and_save_psf_gmm(
        psf8_df,
        "PSF8",
        GMM_GROUPS["PSF8"],
    )
)

[PSF8] Decision_Complexity
 label_map: {1: 0, 2: 1, 0: 2}


,의사결정 대안의 수,의사결정 판단 기준의 수,mean_score
0,1.031122,2.0,1.515561
1,1.006278,0.0,0.503139
2,1.006040,1.0,1.003020


[PSF8] Information_Complexity
 label_map: {0: 0, 2: 1, 1: 2}


,관측되는 정보의 양,(관측이 불가능하여)추론해야하는 정보의 양,mean_score
0,0.0,0.956072,0.478036
1,2.0,0.963025,1.481513
2,1.0,0.991259,0.995629



저장 완료: models\gmm\psf8_gmm.joblib
